**Steps**

1. Load the processed chest radiographs.
2. Link images with study IDs.
3. Separate development and held-out studies.
4. Train a ViT image classifier.
5. Evaluate the image-only baseline.
6. Save the trained ViT.
7. Extract ViT image feature vectors for later multimodal fusion.

### Outputs
- `vit_image_encoder.pt`
- `image_baseline_metrics.csv`
- `image_training_history.csv`
- `image_features_development.csv`
- `image_features_heldout.csv`

In [1]:
import os
import random
import numpy as np
import pandas as pd

import torch
import torch.nn as nn

from torch.utils.data import Dataset, DataLoader

from transformers import (
    ViTImageProcessor,
    ViTModel
)

from sklearn.model_selection import train_test_split

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

from tqdm.auto import tqdm

from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# Paths
base_path = (
    '/content/drive/MyDrive/'
    'dissertation_project/data'
)

processed_path = (
    f'{base_path}/processed'
)

model_path = (
    f'{base_path}/models'
)

os.makedirs(
    model_path,
    exist_ok=True
)

print("Processed path:")
print(processed_path)

print("\nModel path:")
print(model_path)

Processed path:
/content/drive/MyDrive/dissertation_project/data/processed

Model path:
/content/drive/MyDrive/dissertation_project/data/models


In [3]:
# Random seed
SEED = 42


random.seed(SEED)
np.random.seed(SEED)


torch.manual_seed(SEED)


if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)


print(
    "Random seed:",
    SEED
)

Random seed: 42


In [4]:
# Device
device = torch.device(
    'cuda'
    if torch.cuda.is_available()
    else 'cpu'
)


print(
    "Device:",
    device
)


if torch.cuda.is_available():


    print(
        "GPU:",
        torch.cuda.get_device_name(0)
    )

Device: cuda
GPU: Tesla T4


In [5]:
# Load processed images
image_file = (
    f'{processed_path}/processed_images.npz'
)

image_npz = np.load(
    image_file
)

print(
    "Files inside NPZ:"
)

print(
    image_npz.files
)

Files inside NPZ:
['images', 'study_ids']


In [6]:
# Read images and study IDs
images = image_npz['images']


image_study_ids = (
    image_npz['study_ids']
    .astype(str)
)


print(
    "Image array shape:",
    images.shape
)


print(
    "Number of image records:",
    len(images)
)


print(
    "Unique study IDs:",
    len(
        np.unique(
            image_study_ids
        )
    )
)

Image array shape: (2200, 224, 224, 3)
Number of image records: 2200
Unique study IDs: 2200


In [7]:
# Check image values
print(
    "Image dtype:",
    images.dtype
)


print(
    "Minimum value:",
    images.min()
)


print(
    "Maximum value:",
    images.max()
)


print(
    "Image shape:",
    images.shape[1:]
)

Image dtype: float16
Minimum value: 0.0
Maximum value: 1.0
Image shape: (224, 224, 3)


In [8]:
# Load structured labels
structured_file = (
    f'{processed_path}/structured_processed.csv'
)

structured_data = pd.read_csv(
    structured_file
)

structured_data['study_id'] = (
    structured_data['study_id']
    .astype(str)
)

print(
    "Structured data shape:",
    structured_data.shape
)

Structured data shape: (2200, 66)


In [9]:
# Define labels
label_columns = [
    'No Finding',
    'Support Devices',
    'Pleural Effusion',
    'Lung Opacity',
    'Atelectasis',
    'Cardiomegaly',
    'Edema'
]


available_labels = [
    label
    for label in label_columns
    if label in structured_data.columns
]


print(
    "Labels:"
)


print(
    available_labels
)

Labels:
['No Finding', 'Support Devices', 'Pleural Effusion', 'Lung Opacity', 'Atelectasis', 'Cardiomegaly', 'Edema']


In [10]:
# Convert labels to binary
for label in available_labels:

    structured_data[label] = pd.to_numeric(
        structured_data[label],
        errors='coerce'
    ).fillna(0)

    structured_data[label] = (
        structured_data[label] == 1
    ).astype(np.float32)

print(
    "Label values:"
)

for label in available_labels:

    print(
        label,
        sorted(
            structured_data[label]
            .unique()
            .tolist()
        )
    )

Label values:
No Finding [0.0, 1.0]
Support Devices [0.0, 1.0]
Pleural Effusion [0.0, 1.0]
Lung Opacity [0.0, 1.0]
Atelectasis [0.0, 1.0]
Cardiomegaly [0.0, 1.0]
Edema [0.0, 1.0]


In [11]:
# Load held-out IDs
heldout_ids_file = (
    f'{processed_path}/test_calibration_ids.csv'
)


heldout_ids = pd.read_csv(
    heldout_ids_file
)


heldout_ids['study_id'] = (
    heldout_ids['study_id']
    .astype(str)
)


heldout_id_set = set(
    heldout_ids['study_id']
)


print(
    "Held-out studies:",
    len(heldout_id_set)
)

Held-out studies: 687


In [12]:
# Create image metadata table
image_metadata = pd.DataFrame({
    'study_id':
        image_study_ids
})


print(
    image_metadata.head()
)


print(
    "\nImage records:",
    len(image_metadata)
)

   study_id
0  50001166
1  50003120
2  50005491
3  50014721
4  50022513

Image records: 2200


In [13]:
# Attach labels to images
image_metadata = image_metadata.merge(
    structured_data[
        [
            'study_id'
        ] + available_labels
    ],
    on='study_id',
    how='inner'
)


print(
    "Image records with labels:",
    len(image_metadata)
)


print(
    "Unique studies:",
    image_metadata[
        'study_id'
    ].nunique()
)

Image records with labels: 2200
Unique studies: 2200


In [14]:
# Keep only one image per study
image_metadata = (
    image_metadata
    .drop_duplicates(
        subset='study_id'
    )
    .reset_index(drop=True)
)

print(
    "Unique image studies:",
    len(image_metadata)
)

Unique image studies: 2200


In [15]:
# Map image array positions
image_index = {
    study_id: index
    for index, study_id
    in enumerate(image_study_ids)
}


image_metadata['image_index'] = (
    image_metadata['study_id']
    .map(image_index)
)


print(
    image_metadata[
        [
            'study_id',
            'image_index'
        ]
    ].head()
)

   study_id  image_index
0  50001166            0
1  50003120            1
2  50005491            2
3  50014721            3
4  50022513            4


In [16]:
# Separate development and held-out data
image_metadata['is_heldout'] = (
    image_metadata['study_id']
    .isin(heldout_id_set)
)


development_images = image_metadata[
    ~image_metadata['is_heldout']
].copy()


heldout_images = image_metadata[
    image_metadata['is_heldout']
].copy()


development_images = (
    development_images
    .reset_index(drop=True)
)


heldout_images = (
    heldout_images
    .reset_index(drop=True)
)


print(
    "Development images:",
    len(development_images)
)


print(
    "Held-out images:",
    len(heldout_images)
)

Development images: 1513
Held-out images: 687


In [17]:
# Internal training/validation split
train_images, validation_images = train_test_split(
    development_images,
    test_size=0.15,
    random_state=SEED
)

train_images = (
    train_images
    .reset_index(drop=True)
)

validation_images = (
    validation_images
    .reset_index(drop=True)
)

print(
    "Training images:",
    len(train_images)
)

print(
    "Validation images:",
    len(validation_images)
)

Training images: 1286
Validation images: 227


In [18]:
# Load ViT image processor
MODEL_NAME = (
    'google/vit-base-patch16-224-in21k'
)

processor = ViTImageProcessor.from_pretrained(
    MODEL_NAME
)

print(
    "ViT processor loaded."
)

preprocessor_config.json:   0%|          | 0.00/160 [00:00<?, ?B/s]

ViT processor loaded.


In [19]:
# Image dataset
class ViTDataset(Dataset):

    def __init__(
        self,
        dataframe,
        images,
        processor,
        labels
    ):
        self.dataframe = dataframe.reset_index(drop=True)
        self.images = images
        self.processor = processor
        self.labels = labels

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, index):

        row = self.dataframe.iloc[index]

        image_index = int(row['image_index'])

        image = self.images[image_index]

        # Convert image to numpy array
        image = np.asarray(image)

        # Convert to uint8
        if image.dtype != np.uint8:

            image_min = image.min()
            image_max = image.max()

            if image_max > image_min:

                image = (
                    (image - image_min)
                    / (image_max - image_min)
                    * 255.0
                )

            else:

                image = np.zeros_like(
                    image,
                    dtype=np.uint8
                )

            image = image.astype(np.uint8)

        # Convert grayscale image to RGB
        if image.ndim == 2:

            image = np.stack(
                [image, image, image],
                axis=-1
            )

        # Handle channel-first images
        elif image.ndim == 3:

            if (
                image.shape[0] in [1, 3]
                and image.shape[-1] not in [1, 3]
            ):

                image = np.transpose(
                    image,
                    (1, 2, 0)
                )

        # Convert single-channel RGB
        if (
            image.ndim == 3
            and image.shape[-1] == 1
        ):

            image = np.repeat(
                image,
                3,
                axis=-1
            )

        # Ensure RGB
        if (
            image.ndim == 3
            and image.shape[-1] != 3
        ):

            image = image[:, :, :3]

        # ViT preprocessing
        encoded = self.processor(
            images=image,
            return_tensors='pt'
        )

        pixel_values = encoded[
            'pixel_values'
        ].squeeze(0)

        # Labels
        labels = np.array(
            [
                row[label]
                for label in self.labels
            ],
            dtype=np.float32
        )

        labels = torch.tensor(
            labels,
            dtype=torch.float32
        )

        return {
            'pixel_values': pixel_values,
            'labels': labels
        }


print("ViTDataset class defined successfully.")

ViTDataset class defined successfully.


In [20]:
print(ViTDataset)

print(
    "Number of training rows:",
    len(train_images)
)

test_dataset = ViTDataset(
    train_images.iloc[:2].copy(),
    images,
    processor,
    available_labels
)

print(
    "Test dataset length:",
    len(test_dataset)
)

<class '__main__.ViTDataset'>
Number of training rows: 1286
Test dataset length: 2


In [21]:
# Create datasets

train_dataset = ViTDataset(
    train_images,
    images,
    processor,
    available_labels
)

validation_dataset = ViTDataset(
    validation_images,
    images,
    processor,
    available_labels
)

print(
    "Training dataset:",
    len(train_dataset)
)

print(
    "Validation dataset:",
    len(validation_dataset)
)

Training dataset: 1286
Validation dataset: 227


In [22]:
# DataLoaders
BATCH_SIZE = 16


train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)


validation_loader = DataLoader(
    validation_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)


print(
    "Training batches:",
    len(train_loader)
)


print(
    "Validation batches:",
    len(validation_loader)
)

Training batches: 81
Validation batches: 15


In [23]:
# Test one image
sample = train_dataset[0]

print(
    "Pixel tensor shape:",
    sample[
        'pixel_values'
    ].shape
)

print(
    "Labels:",
    sample[
        'labels'
    ]
)

Pixel tensor shape: torch.Size([3, 224, 224])
Labels: tensor([1., 0., 0., 0., 0., 0., 0.])


In [24]:
# Define ViT classifier
class ViTClassifier(nn.Module):


    def __init__(
        self,
        model_name,
        num_labels
    ):


        super().__init__()


        self.vit = ViTModel.from_pretrained(
            model_name
        )


        hidden_size = (
            self.vit.config.hidden_size
        )


        self.dropout = nn.Dropout(
            0.2
        )


        self.classifier = nn.Linear(
            hidden_size,
            num_labels
        )


    def forward(
        self,
        pixel_values
    ):


        outputs = self.vit(
            pixel_values=pixel_values
        )


        # CLS token
        features = (
            outputs.last_hidden_state[:, 0, :]
        )


        features = self.dropout(
            features
        )


        logits = self.classifier(
            features
        )


        return logits, features

In [25]:
# Create model
model = ViTClassifier(
    MODEL_NAME,
    len(available_labels)
)


model = model.to(device)


print(
    "ViT model loaded."
)


print(
    "Hidden feature size:",
    model.vit.config.hidden_size
)

config.json:   0%|          | 0.00/502 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  346MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

ViT model loaded.
Hidden feature size: 768


In [26]:
# Class weights
y_train = (
    train_images[
        available_labels
    ]
    .values
    .astype(np.float32)
)


positive_counts = (
    y_train.sum(axis=0)
)


negative_counts = (
    len(y_train)
    - positive_counts
)


pos_weights = (
    negative_counts
    /
    np.maximum(
        positive_counts,
        1
    )
)


pos_weights = torch.tensor(
    pos_weights,
    dtype=torch.float32
).to(device)


print(
    pd.DataFrame({
        'label': available_labels,
        'positive': positive_counts,
        'negative': negative_counts,
        'weight':
            pos_weights.cpu().numpy()
    })
)

              label  positive  negative    weight
0        No Finding     340.0     946.0  2.782353
1   Support Devices     466.0     820.0  1.759657
2  Pleural Effusion     387.0     899.0  2.322997
3      Lung Opacity     351.0     935.0  2.663818
4       Atelectasis     314.0     972.0  3.095541
5      Cardiomegaly     337.0     949.0  2.816024
6             Edema     192.0    1094.0  5.697917


In [27]:
# Loss and optimizer
criterion = nn.BCEWithLogitsLoss(
    pos_weight=pos_weights
)


optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=2e-5,
    weight_decay=0.01
)


print(
    "Optimizer ready."
)

Optimizer ready.


In [28]:
# Training function
def train_one_epoch(
    model,
    loader,
    optimizer,
    criterion,
    device
):


    model.train()


    total_loss = 0


    progress = tqdm(
        loader,
        desc='Training'
    )


    for batch in progress:


        pixel_values = (
            batch['pixel_values']
            .to(
                device,
                non_blocking=True
            )
        )


        labels = (
            batch['labels']
            .to(device)
        )


        optimizer.zero_grad()


        logits, _ = model(
            pixel_values
        )


        loss = criterion(
            logits,
            labels
        )


        loss.backward()


        optimizer.step()


        total_loss += (
            loss.item()
            * pixel_values.size(0)
        )


        progress.set_postfix(
            loss=loss.item()
        )


    return (
        total_loss
        /
        len(loader.dataset)
    )

In [29]:
# Evaluation function
def evaluate_model(
    model,
    loader,
    criterion,
    device
):


    model.eval()


    total_loss = 0


    all_labels = []
    all_probabilities = []


    with torch.no_grad():


        for batch in loader:


            pixel_values = (
                batch['pixel_values']
                .to(
                    device,
                    non_blocking=True
                )
            )


            labels = (
                batch['labels']
                .to(device)
            )


            logits, _ = model(
                pixel_values
            )


            probabilities = (
                torch.sigmoid(logits)
            )


            loss = criterion(
                logits,
                labels
            )


            total_loss += (
                loss.item()
                * pixel_values.size(0)
            )


            all_labels.append(
                labels.cpu().numpy()
            )


            all_probabilities.append(
                probabilities.cpu().numpy()
            )


    y_true = np.vstack(
        all_labels
    )


    y_prob = np.vstack(
        all_probabilities
    )


    y_pred = (
        y_prob >= 0.5
    ).astype(int)


    return (
        total_loss / len(loader.dataset),
        y_true,
        y_prob,
        y_pred
    )

In [30]:
# Train the ViT
EPOCHS = 2

history = []

for epoch in range(EPOCHS):

    print(
        f"\n========== "
        f"Epoch {epoch + 1}/{EPOCHS}"
        f" =========="
    )

    train_loss = train_one_epoch(
        model,
        train_loader,
        optimizer,
        criterion,
        device
    )

    val_loss, y_true, y_prob, y_pred = (
        evaluate_model(
            model,
            validation_loader,
            criterion,
            device
        )
    )

    print(
        f"Training loss: "
        f"{train_loss:.4f}"
    )

    print(
        f"Validation loss: "
        f"{val_loss:.4f}"
    )

    history.append({
        'epoch': epoch + 1,
        'train_loss': train_loss,
        'validation_loss': val_loss
    })


========== Epoch 1/2 ==========


Training:   0%|          | 0/81 [00:00<?, ?it/s]

Training loss: 0.9826
Validation loss: 0.9250

========== Epoch 2/2 ==========


Training:   0%|          | 0/81 [00:00<?, ?it/s]

Training loss: 0.9126
Validation loss: 0.8968


In [31]:
# Training history
history_df = pd.DataFrame(
    history
)


display(
    history_df
)

,epoch,train_loss,validation_loss
0,1,0.982631,0.925012
1,2,0.912553,0.896799


In [32]:
# Calculate image baseline metrics
metrics = []


for i, label in enumerate(
    available_labels
):


    true_values = (
        y_true[:, i]
        == 1
    ).astype(int)


    probabilities = (
        y_prob[:, i]
    )


    predicted_values = (
        probabilities >= 0.5
    ).astype(int)


    accuracy = accuracy_score(
        true_values,
        predicted_values
    )


    precision = precision_score(
        true_values,
        predicted_values,
        average='binary',
        zero_division=0
    )


    recall = recall_score(
        true_values,
        predicted_values,
        average='binary',
        zero_division=0
    )


    f1 = f1_score(
        true_values,
        predicted_values,
        average='binary',
        zero_division=0
    )


    try:


        roc_auc = roc_auc_score(
            true_values,
            probabilities
        )


    except ValueError:


        roc_auc = np.nan


    metrics.append({
        'label': label,
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'roc_auc': roc_auc
    })


image_baseline_metrics = pd.DataFrame(
    metrics
)


display(
    image_baseline_metrics
)

,label,accuracy,precision,recall,f1,roc_auc
0,No Finding,0.814978,0.666667,0.727273,0.695652,0.795502
1,Support Devices,0.647577,0.550000,0.819149,0.658120,0.729403
2,Pleural Effusion,0.585903,0.367647,0.862069,0.515464,0.797796
3,Lung Opacity,0.555066,0.375000,0.904762,0.530233,0.708188
4,Atelectasis,0.568282,0.351145,0.779661,0.484211,0.697437
5,Cardiomegaly,0.568282,0.333333,0.849057,0.478723,0.675884
6,Edema,0.528634,0.188976,0.857143,0.309677,0.669957


In [33]:
# Overall image baseline
y_true_binary = (
    y_true == 1
).astype(int)


y_pred_binary = (
    y_prob >= 0.5
).astype(int)




overall_image_metrics = pd.DataFrame({
    'metric': [
        'Accuracy',
        'Precision',
        'Recall',
        'F1',
        'ROC-AUC'
    ],


    'value': [


        accuracy_score(
            y_true_binary.flatten(),
            y_pred_binary.flatten()
        ),


        precision_score(
            y_true_binary.flatten(),
            y_pred_binary.flatten(),
            average='binary',
            zero_division=0
        ),


        recall_score(
            y_true_binary.flatten(),
            y_pred_binary.flatten(),
            average='binary',
            zero_division=0
        ),


        f1_score(
            y_true_binary.flatten(),
            y_pred_binary.flatten(),
            average='binary',
            zero_division=0
        ),


        roc_auc_score(
            y_true_binary,
            y_prob,
            average='macro'
        )
    ]
})


display(
    overall_image_metrics
)

,metric,value
0,Accuracy,0.609817
1,Precision,0.388578
2,Recall,0.824228
3,F1,0.528158
4,ROC-AUC,0.724881


In [34]:
# Save metrics
metrics_file = (
    f'{processed_path}/'
    'image_baseline_metrics.csv'
)


image_baseline_metrics.to_csv(
    metrics_file,
    index=False
)


print(
    "Saved:"
)


print(
    metrics_file
)

Saved:
/content/drive/MyDrive/dissertation_project/data/processed/image_baseline_metrics.csv


In [35]:
# Save training history
history_file = (
    f'{processed_path}/'
    'image_training_history.csv'
)


history_df.to_csv(
    history_file,
    index=False
)


print(
    "Saved:"
)


print(
    history_file
)

Saved:
/content/drive/MyDrive/dissertation_project/data/processed/image_training_history.csv


In [36]:
# Save trained ViT
model_file = (
    f'{model_path}/'
    'vit_image_encoder.pt'
)


torch.save(
    {
        'model_state_dict':
            model.state_dict(),


        'model_name':
            MODEL_NAME,


        'labels':
            available_labels
    },
    model_file
)


print(
    "Saved model:"
)


print(
    model_file
)

Saved model:
/content/drive/MyDrive/dissertation_project/data/models/vit_image_encoder.pt


In [37]:
# Feature extraction function
def extract_image_features(
    model,
    dataframe,
    images,
    processor,
    device,
    batch_size=16
):

    dataset = ViTDataset(
        dataframe,
        images,
        processor,
        available_labels
    )

    loader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=2,
        pin_memory=True
    )

    model.eval()

    feature_list = []

    with torch.no_grad():

        for batch in tqdm(
            loader,
            desc='Extracting image features'
        ):

            pixel_values = (
                batch['pixel_values']
                .to(
                    device,
                    non_blocking=True
                )
            )

            _, features = model(
                pixel_values
            )

            feature_list.append(
                features.cpu().numpy()
            )

    return np.vstack(
        feature_list
    )

In [38]:
# Extract development image features
development_features = (
    extract_image_features(
        model,
        development_images,
        images,
        processor,
        device,
        BATCH_SIZE
    )
)


print(
    "Development feature shape:",
    development_features.shape
)

Extracting image features:   0%|          | 0/95 [00:00<?, ?it/s]

Development feature shape: (1513, 768)


In [39]:
# Extract held-out image features
heldout_features = (
    extract_image_features(
        model,
        heldout_images,
        images,
        processor,
        device,
        BATCH_SIZE
    )
)


print(
    "Held-out feature shape:",
    heldout_features.shape
)

Extracting image features:   0%|          | 0/43 [00:00<?, ?it/s]

Held-out feature shape: (687, 768)


In [40]:
# Save development image features
development_feature_df = pd.DataFrame(
    development_features,
    columns=[
        f'image_feature_{i}'
        for i in range(
            development_features.shape[1]
        )
    ]
)


development_feature_df.insert(
    0,
    'study_id',
    development_images[
        'study_id'
    ].values
)


development_feature_file = (
    f'{processed_path}/'
    'image_features_development.csv'
)


development_feature_df.to_csv(
    development_feature_file,
    index=False
)


print(
    "Saved:"
)


print(
    development_feature_file
)

Saved:
/content/drive/MyDrive/dissertation_project/data/processed/image_features_development.csv


In [41]:
# Save held-out image features
heldout_feature_df = pd.DataFrame(
    heldout_features,
    columns=[
        f'image_feature_{i}'
        for i in range(
            heldout_features.shape[1]
        )
    ]
)


heldout_feature_df.insert(
    0,
    'study_id',
    heldout_images[
        'study_id'
    ].values
)


heldout_feature_file = (
    f'{processed_path}/'
    'image_features_heldout.csv'
)


heldout_feature_df.to_csv(
    heldout_feature_file,
    index=False
)


print(
    "Saved:"
)


print(
    heldout_feature_file
)

Saved:
/content/drive/MyDrive/dissertation_project/data/processed/image_features_heldout.csv
